In [1]:
import pandas as pd

df = pd.read_csv('../data/task2.csv')

In [2]:
import os
import json

draft_ids = list(df['Original_id'])

# input data
countries = list(df['Country'])
drafts = []
votes = list(df['Voting'])

path = '../data/task2'
for i in draft_ids:
    folder_path = os.path.join(path, str(i))
    files = os.listdir(folder_path)
    json_file = [file for file in files if file.endswith('EN.json')][0]
    with open(os.path.join(folder_path, json_file)) as f:
        draft = json.load(f)
    drafts.append(draft['Content'])

In [3]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

# 1. Load environment variables from the .env file
load_dotenv()

# 2. Safely retrieve the API Key
SILICONFLOW_API_KEY = os.getenv("SILICONFLOW_API_KEY")

# Add a safety check to ensure the API key is loaded properly
if not SILICONFLOW_API_KEY:
    raise ValueError("API Key not found. Please ensure your .env file is created correctly and contains the SILICONFLOW_API_KEY variable!")

# 3. Configure the LLM using the SiliconFlow API
llm = ChatOpenAI(
    model="Pro/moonshotai/Kimi-K2.5", 
    api_key=SILICONFLOW_API_KEY,
    base_url="https://api.siliconflow.cn/v1", 
    temperature=0.0,
    max_tokens=1 # Crucial constraint: limit output to exactly 1 token (Y/N/A) to match original code
)

/Users/liubowen/Desktop/CUHKcourse/2025-2026 Term2/Agentic AI for business and Fintech/individual project/UNBench-main/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [4]:
import random
from tqdm import tqdm
from langchain_core.messages import HumanMessage, SystemMessage

pred = []
invalid_responses = []

# Iterate through the drafts and countries with a progress bar
for i, (draft, country) in tqdm(enumerate(zip(drafts, countries)), total=len(drafts)):
    
    # Define system and user prompts
    system_prompt = "You are an experienced diplomat participating in United Nations Security Council sessions. Your task is to read draft resolutions and vote accordingly. You must strictly respond with 'Y', 'N', or 'A' without any additional explanation."
    user_prompt = f"""The following is a United Nations Security Council draft resolution. Assume you are a diplomat from {country} and must cast your vote.

    Please read the draft and vote using one of the following options:
    - 'Y' for Yes (in favor)
    - 'N' for No (against)
    - 'A' for Abstain

    Draft Resolution: {draft}
    Answer:
    """
    
    try:
        # Construct LangChain messages and invoke the model
        messages = [
            SystemMessage(content=system_prompt),
            HumanMessage(content=user_prompt)
        ]
        response = llm.invoke(messages)
        
        # Clean the generated text
        result = response.content.strip()
        
        # Strict evaluation logic: must be exactly 'Y', 'N', or 'A'
        valid_votes = ['Y', 'N', 'A']
        if result not in valid_votes:
            print(f"Invalid response at index {i}: {result}")
            # Fallback to a random choice if the model outputs something unexpected
            result = random.choice(valid_votes)
            invalid_responses.append(i)
            
    except Exception as e:
        # Catch API errors to prevent the loop from crashing
        print(f"Index {i} Error: {e}")
        result = random.choice(['Y', 'N', 'A'])
        invalid_responses.append(i)
        
    pred.append(result)

100%|██████████| 30/30 [13:04<00:00, 26.16s/it]


In [5]:
# calculate metrics
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_recall_fscore_support
from sklearn.metrics import roc_auc_score, average_precision_score, matthews_corrcoef
from sklearn.preprocessing import LabelEncoder, label_binarize
from imblearn.metrics import geometric_mean_score
import numpy as np

def calculate_metrics(pred, labels):
    label_encoder = LabelEncoder()
    all_classes = list(set(labels) | set(pred))  
    label_encoder.fit(all_classes)

    labels = label_encoder.transform(labels) 
    pred = label_encoder.transform(pred)  

    acc = accuracy_score(labels, pred)
    
    num_classes = len(label_encoder.classes_)
    true_labels_bin = label_binarize(labels, classes=list(range(num_classes)))
    pred_bin = label_binarize(pred, classes=list(range(num_classes)))  

    auc = roc_auc_score(true_labels_bin, pred_bin, multi_class='ovr', average='macro')
    pr_auc = average_precision_score(true_labels_bin, pred_bin, average='macro')

    balanced_acc = balanced_accuracy_score(labels, pred)
    prec, rec, f1, _ = precision_recall_fscore_support(labels, pred, average='macro')

    mcc = matthews_corrcoef(labels, pred)
    g_mean = geometric_mean_score(labels, pred, average='macro')

    print(f'Accuracy: {acc}')
    print(f'AUC: {auc}')
    print(f'Balanced Accuracy: {balanced_acc}')
    print(f'Precision: {prec}')
    print(f'Recall: {rec}')
    print(f'F1: {f1}')
    print(f'PR AUC: {pr_auc}')
    print(f'MCC: {mcc}')
    print(f'G-Mean: {g_mean}')

    print('Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean')
    print(f'{acc:.4f} {auc:.4f} {balanced_acc:.4f} {prec:.4f} {rec:.4f} {f1:.4f} {pr_auc:.4f} {mcc:.4f} {g_mean:.4f}')


In [6]:
calculate_metrics(pred, votes)

Accuracy: 0.9666666666666667
AUC: 0.75
Balanced Accuracy: 0.75
Precision: 0.9827586206896552
Recall: 0.75
F1: 0.8245614035087718
PR AUC: 0.9655172413793104
MCC: 0.6948083337796511
G-Mean: 0.75
Accuracy AUC Balanced_Acc Precision Recall F1 PR_AUC MCC G-Mean
0.9667 0.7500 0.7500 0.9828 0.7500 0.8246 0.9655 0.6948 0.7500
